# Axial v3 Iteration B - low-cost runner

Orquestador B0-B6 reproducible. No ejecuta todos los experimentos por defecto.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", ".")).resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
from lumbar_mri.axial_v3.experiments import estimate_run_count, load_low_cost_grid, select_experiment
from lumbar_mri.axial_v3.guards import require_train_val_only
from lumbar_mri.axial_v3.training import AxialV3TrainConfig, build_train_val_loaders, compute_class_weights, run_training

ALLOWED_SPLITS = require_train_val_only(["train", "val"], context="notebook_52")
CONFIG_PATH = REPO_ROOT / "config" / "axial_v3_low_cost_grid.json"
GRID = load_low_cost_grid(CONFIG_PATH)
RUN_MODE = os.getenv("RUN_MODE", "preflight")
EXPERIMENT_ID = os.getenv("PFI_AXIAL_V3_EXPERIMENT_ID", "B0")

In [ ]:
def train_config_for_experiment(experiment_id):
    experiment = select_experiment(GRID, experiment_id)
    return replace(
        AxialV3TrainConfig(),
        RUN_ID=experiment.runId,
        RAW0_WEIGHT_BOOST=experiment.raw0EffectiveWeightMultiplier,
        MAX_CLASS_WEIGHT_RATIO=experiment.maxClassWeightRatio,
        MAX_EPOCHS=experiment.maxEpochs,
        EARLY_STOPPING_PATIENCE=experiment.earlyStoppingPatience,
        PRESENCE_HEAD_ENABLED=experiment.presenceHeadEnabled,
        LAMBDA_PRESENCE=experiment.lambdaPresence,
        RAW0_BALANCED_SAMPLER_ENABLED=experiment.raw0BalancedSamplerEnabled,
        POSITIVE_FRACTION=experiment.positiveFraction or 0.5,
    )


CFG = train_config_for_experiment(EXPERIMENT_ID)

In [ ]:
RUN_EXPERIMENT = os.getenv("PFI_RUN_AXIAL_V3_EXPERIMENT", "0") == "1"

if RUN_MODE == "preflight":
    print(json.dumps({"mode": RUN_MODE, "experimentId": EXPERIMENT_ID, "plannedRuns": estimate_run_count(GRID), "runId": CFG.RUN_ID}, indent=2))
elif RUN_MODE == "smoke" and RUN_EXPERIMENT:
    print(json.dumps(run_training(CFG, smoke=True), indent=2, default=str))
elif RUN_MODE == "train" and RUN_EXPERIMENT:
    print(json.dumps(run_training(CFG, smoke=False), indent=2, default=str))
elif RUN_MODE == "summarize":
    print("Summarize lee solo reportes validation de Iteracion B; no carga datos de evaluacion final.")
else:
    print("Iteration B preparada. Definir RUN_MODE y PFI_RUN_AXIAL_V3_EXPERIMENT=1 para ejecutar.")